In [1]:
# ========================================================================
# REVENUE BY GENRE ANALYSIS - STEAM GAMES
# ========================================================================
# This script creates a revenue proxy and analyzes revenue by game genre
# Author: Cedric
# Date: December 2025
# ========================================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Set display options for better readability
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', 50)

In [2]:
# ========================================================================
# 1. LOAD DATA
# ========================================================================

print("=" * 70)
print("LOADING STEAM GAMES DATASET")
print("=" * 70)

# Load cleaned dataset
df = pd.read_csv('steam_games_cleaned.csv')

print(f"✓ Dataset loaded successfully!")
print(f"Shape: {df.shape[0]:,} games × {df.shape[1]} columns")
print(f"\nColumns available: {df.columns.tolist()}")

LOADING STEAM GAMES DATASET
✓ Dataset loaded successfully!
Shape: 42,497 games × 35 columns

Columns available: ['app_id', 'title', 'release_date', 'release_year', 'game_age_years', 'age_bucket', 'genres_clean', 'main_genre', 'categories_clean', 'developer', 'publisher', 'original_price_clean', 'discount_pct_clean', 'discounted_price_clean', 'final_price', 'is_free_to_play', 'is_on_sale', 'dlc_available', 'has_dlc', 'age_rating', 'age_rating_category', 'win_support', 'mac_support', 'linux_support', 'multi_platform', 'awards', 'is_early_access', 'overall_review', 'overall_review_pct', 'overall_review_count', 'log_overall_review_count', 'recent_review', 'recent_review_pct', 'recent_review_count', 'log_recent_review_count']


In [3]:
# ========================================================================
# 2. CREATE REVENUE PROXY
# ========================================================================

print("\n" + "=" * 70)
print("CREATING REVENUE PROXY")
print("=" * 70)

# Industry standard: approximately 1-2% of players leave reviews
# Conservative estimate: 1 review ≈ 50-100 owners
REVIEW_TO_OWNER_RATIO = 75  # Conservative middle ground

print(f"\nAssumptions:")
print(f"• Review-to-Owner Ratio: 1 review = {REVIEW_TO_OWNER_RATIO} owners")
print(f"• F2P Revenue per Player: ₹50 (conservative in-app purchase estimate)")

# Create estimated owners column
df['estimated_owners'] = df['overall_review_count'] * REVIEW_TO_OWNER_RATIO

# Create revenue proxy for paid games
df['estimated_revenue'] = df['estimated_owners'] * df['final_price']

# For free-to-play games, estimate revenue differently (in-app purchases)
# Assume F2P generates ~₹50 per engaged player (conservative)
F2P_REVENUE_PER_PLAYER = 50

df.loc[df['is_free_to_play'] == True, 'estimated_revenue'] = (
    df.loc[df['is_free_to_play'] == True, 'estimated_owners'] * F2P_REVENUE_PER_PLAYER
)

# Display summary statistics
print(f"\n✓ Revenue Proxy Created!")
print(f"\nRevenue Statistics:")
print(f"  Total estimated revenue: ₹{df['estimated_revenue'].sum()/1e9:.2f} Billion")
print(f"  Median revenue per game: ₹{df['estimated_revenue'].median()/1e6:.2f} Million")
print(f"  Mean revenue per game: ₹{df['estimated_revenue'].mean()/1e6:.2f} Million")
print(f"  Games with revenue data: {df['estimated_revenue'].notna().sum():,}")


CREATING REVENUE PROXY

Assumptions:
• Review-to-Owner Ratio: 1 review = 75 owners
• F2P Revenue per Player: ₹50 (conservative in-app purchase estimate)

✓ Revenue Proxy Created!

Revenue Statistics:
  Total estimated revenue: ₹6004.71 Billion
  Median revenue per game: ₹0.95 Million
  Mean revenue per game: ₹150.91 Million
  Games with revenue data: 39,790


In [4]:
# ========================================================================
# 3. TOP REVENUE GAMES
# ========================================================================

print("\n" + "=" * 70)
print("TOP 10 GAMES BY ESTIMATED REVENUE")
print("=" * 70)

top_revenue = df.nlargest(10, 'estimated_revenue')[
    ['title', 'main_genre', 'final_price', 'overall_review_count',
     'estimated_owners', 'estimated_revenue', 'is_free_to_play']
].copy()

# Format for display
top_revenue['estimated_revenue_millions'] = (
    top_revenue['estimated_revenue'] / 1e6
).round(2)
top_revenue['estimated_owners_k'] = (
    top_revenue['estimated_owners'] / 1000
).round(0).astype(int)

print("\n")
print(top_revenue[
    ['title', 'main_genre', 'final_price', 'estimated_owners_k',
     'estimated_revenue_millions', 'is_free_to_play']
].to_string(index=False))


TOP 10 GAMES BY ESTIMATED REVENUE


                                    title main_genre  final_price  estimated_owners_k  estimated_revenue_millions  is_free_to_play
                               ELDEN RING     action       3599.0               45389                   163356.18            False
                    Red Dead Redemption 2     action       3199.0               39545                   126505.97            False
                          Baldur's Gate 3  adventure       2999.0               40731                   122151.59            False
                            HELLDIVERS™ 2     action       2499.0               47913                   119735.71            False
                                     Rust     action       1799.0               65472                   117784.13            False
                       Grand Theft Auto V     action        952.0              123018                   117112.71            False
          Tom Clancy's Rainbow Six® Siege     

In [5]:
# ========================================================================
# 4. REVENUE BY GENRE ANALYSIS
# ========================================================================

print("\n" + "=" * 70)
print("REVENUE BY GENRE - COMPREHENSIVE ANALYSIS")
print("=" * 70)

# Calculate revenue metrics by main_genre
genre_revenue = df.groupby('main_genre').agg({
    'estimated_revenue': ['sum', 'median', 'mean'],
    'app_id': 'count',
    'overall_review_count': 'median',
    'final_price': 'median'
}).round(2)

# Flatten column names
genre_revenue.columns = [
    'total_revenue', 'median_revenue', 'mean_revenue',
    'game_count', 'median_reviews', 'median_price'
]

# Convert to millions/billions for readability
genre_revenue['total_revenue_billion'] = (
    genre_revenue['total_revenue'] / 1e9
).round(2)
genre_revenue['median_revenue_million'] = (
    genre_revenue['median_revenue'] / 1e6
).round(2)
genre_revenue['mean_revenue_million'] = (
    genre_revenue['mean_revenue'] / 1e6
).round(2)

# Sort by total revenue
genre_revenue = genre_revenue.sort_values('total_revenue', ascending=False)

print("\nTop 15 Genres by Total Estimated Revenue:\n")
top_genres = genre_revenue.head(15)[
    ['game_count', 'total_revenue_billion', 'median_revenue_million',
     'mean_revenue_million', 'median_price']
]
print(top_genres.to_string())

# Calculate market share
total_market_revenue = df['estimated_revenue'].sum()
genre_revenue['market_share_pct'] = (
    genre_revenue['total_revenue'] / total_market_revenue * 100
).round(2)

print("\n" + "=" * 70)
print("MARKET SHARE BY GENRE (Top 10)")
print("=" * 70)
market_share = genre_revenue.head(10)[
    ['total_revenue_billion', 'market_share_pct', 'game_count']
]
print(market_share.to_string())


REVENUE BY GENRE - COMPREHENSIVE ANALYSIS

Top 15 Genres by Total Estimated Revenue:

                       game_count  total_revenue_billion  median_revenue_million  mean_revenue_million  median_price
main_genre                                                                                                          
action                      18170                4267.00                    0.98                248.15         259.0
adventure                    9570                 535.54                    1.07                 59.88         299.0
rpg                           705                 292.02                    7.40                445.84         480.0
simulation                   1110                 277.56                    3.32                280.65         459.0
indie                        4145                 256.58                    1.26                 65.65         300.0
strategy                      731                 144.72                    3.90              

In [6]:
# ========================================================================
# 5. REVENUE EFFICIENCY ANALYSIS
# ========================================================================

print("\n" + "=" * 70)
print("REVENUE EFFICIENCY ANALYSIS BY GENRE")
print("=" * 70)

# Calculate revenue per game (efficiency metric)
efficiency = genre_revenue.copy()
efficiency['revenue_per_game_million'] = (
    efficiency['total_revenue'] / efficiency['game_count'] / 1e6
).round(2)

# Sort by revenue per game
efficiency_sorted = efficiency.sort_values(
    'revenue_per_game_million',
    ascending=False
).head(15)

print("\nTop 15 Genres by Revenue per Game (Efficiency):\n")
print(efficiency_sorted[
    ['game_count', 'revenue_per_game_million', 'median_price', 'median_reviews']
].to_string())



REVENUE EFFICIENCY ANALYSIS BY GENRE

Top 15 Genres by Revenue per Game (Efficiency):

                       game_count  revenue_per_game_million  median_price  median_reviews
main_genre                                                                               
rpg                           705                    414.22         480.0           187.0
simulation                   1110                    250.06         459.0           115.0
racing                        287                    244.77         480.0           231.5
action                      18170                    234.84         259.0            59.0
strategy                      731                    197.98         369.0           141.0
massively multiplayer          77                    139.85           0.0           283.0
indie                        4145                     61.90         300.0            68.0
adventure                    9570                     55.96         299.0            60.0
sports      

In [7]:
# ========================================================================
# 6. KEY INSIGHTS
# ========================================================================

print("\n" + "=" * 70)
print("KEY INSIGHTS - REVENUE BY GENRE")
print("=" * 70)

insights = """

1. MARKET DOMINANCE - Action Genre Rules
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
   • Action games: ₹4,267 Billion (71.06% market share)
   • 18,170 action games generating 71% of total revenue
   • More than 8x the revenue of #2 genre (Adventure)

2. HIGH-VALUE GENRES - Quality over Quantity
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
   • RPG: ₹414M average per game (only 705 games, 4.86% market share)
   • Simulation: ₹250M average per game (1,110 games)
   • Racing: ₹245M average per game (287 games)
   → These genres have fewer titles but higher engagement & pricing

3. VOLUME vs VALUE STRATEGY
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
   • Casual: 7,017 games but only ₹20M avg revenue per game
   • Indie: 4,145 games with ₹62M avg revenue per game
   • Action: 18,170 games with ₹235M avg revenue per game
   → Action succeeds through BOTH volume AND value

4. PRICING CORRELATION
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
   • High revenue genres have higher median prices:
     - RPG: ₹480 median price
     - Racing: ₹480 median price
     - Simulation: ₹459 median price
   • Lower revenue genres have lower prices:
     - Casual: ₹155 median price
     - Free-to-play: ₹0 median price

5. FREE-TO-PLAY PARADOX
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
   • F2P: Only ₹1.25 Billion total (0.02% market share)
   • 485 games with ₹2.6M average revenue per game
   • Despite massive player counts (Counter-Strike, Dota 2)
   → Revenue proxy underestimates F2P (we used conservative ₹50/player)
   → Real F2P revenue likely much higher with in-app purchases

BUSINESS RECOMMENDATIONS:
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

For Developers:
• Target Action, RPG, or Simulation for highest revenue potential
• Premium pricing (₹400-500+) correlates with better revenue
• RPG/Racing have best revenue-per-game ratios despite smaller catalogs

For Platform (Steam):
• Action genre drives 71% of revenue - optimize discovery/promotion
• Support premium genres (RPG, Sim, Racing) - high value per title
• Casual/Indie saturated markets - harder for new entrants
"""

print(insights)


KEY INSIGHTS - REVENUE BY GENRE


1. MARKET DOMINANCE - Action Genre Rules
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
   • Action games: ₹4,267 Billion (71.06% market share)
   • 18,170 action games generating 71% of total revenue
   • More than 8x the revenue of #2 genre (Adventure)

2. HIGH-VALUE GENRES - Quality over Quantity
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
   • RPG: ₹414M average per game (only 705 games, 4.86% market share)
   • Simulation: ₹250M average per game (1,110 games)
   • Racing: ₹245M average per game (287 games)
   → These genres have fewer titles but higher engagement & pricing

3. VOLUME vs VALUE STRATEGY
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
   • Casual: 7,017 games but only ₹20M avg revenue per game
   • Indie: 4,145 games with ₹62M avg revenue per game
   • Action: 18,170 games with ₹235M avg revenue per game
   → Action succeeds through BOTH volume AND value

4. PRICING CORRELATION
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
   • High revenue genres h

In [8]:
# ========================================================================
# 7. EXECUTIVE SUMMARY
# ========================================================================

total_revenue = genre_revenue['total_revenue'].sum()
summary_df = pd.DataFrame({
    'Metric': [
        'Total Market Revenue',
        'Number of Genres Analyzed',
        'Top Genre by Revenue',
        'Top Genre Market Share',
        'Most Efficient Genre (Rev/Game)',
        'Largest Genre by Count',
        'Highest Median Price Genre'
    ],
    'Value': [
        f"₹{total_revenue/1e9:.2f} Billion",
        f"{len(genre_revenue)}",
        'Action',
        '71.06%',
        'RPG (₹414M/game)',
        'Action (18,170 games)',
        'RPG (₹480)'
    ]
})

print("\n" + "=" * 70)
print("EXECUTIVE SUMMARY - REVENUE BY GENRE")
print("=" * 70)
print(summary_df.to_string(index=False))


EXECUTIVE SUMMARY - REVENUE BY GENRE
                         Metric                 Value
           Total Market Revenue      ₹6000.61 Billion
      Number of Genres Analyzed                    13
           Top Genre by Revenue                Action
         Top Genre Market Share                71.06%
Most Efficient Genre (Rev/Game)      RPG (₹414M/game)
         Largest Genre by Count Action (18,170 games)
     Highest Median Price Genre            RPG (₹480)


In [9]:
# ========================================================================
# 8. SAVE RESULTS
# ========================================================================

print("\n" + "=" * 70)
print("SAVING RESULTS")
print("=" * 70)

# Save genre revenue analysis
genre_revenue.to_csv('genre_revenue_analysis.csv')
print("✓ Saved: genre_revenue_analysis.csv")

# Save top revenue games
top_revenue.to_csv('top_revenue_games.csv', index=False)
print("✓ Saved: top_revenue_games.csv")

# Save executive summary
summary_df.to_csv('revenue_executive_summary.csv', index=False)
print("✓ Saved: revenue_executive_summary.csv")


SAVING RESULTS
✓ Saved: genre_revenue_analysis.csv
✓ Saved: top_revenue_games.csv
✓ Saved: revenue_executive_summary.csv


In [10]:
# ========================================================================
# 9. VISUALIZATIONS (using matplotlib/seaborn)
# ========================================================================

print("\n" + "=" * 70)
print("CREATING VISUALIZATIONS")
print("=" * 70)

# Set style
sns.set(style="whitegrid", palette="muted")
plt.rcParams['figure.figsize'] = (12, 8)

# Visualization 1: Total Revenue by Genre (Horizontal Bar Chart)
fig, ax = plt.subplots(figsize=(12, 8))

top_15_genres = genre_revenue.head(15).copy()
top_15_genres = top_15_genres.sort_values('total_revenue_billion')

ax.barh(
    top_15_genres.index,
    top_15_genres['total_revenue_billion'],
    color='steelblue'
)

ax.set_xlabel('Estimated Revenue (₹ Billions)', fontsize=12, fontweight='bold')
ax.set_ylabel('Genre', fontsize=12, fontweight='bold')
ax.set_title(
    'Total Estimated Revenue by Genre (Top 15)',
    fontsize=14,
    fontweight='bold',
    pad=20
)

# Add value labels
for i, v in enumerate(top_15_genres['total_revenue_billion']):
    ax.text(v + 50, i, f'₹{v:.0f}B', va='center', fontsize=10)

plt.tight_layout()
plt.savefig('revenue_by_genre_bar_chart.png', dpi=300, bbox_inches='tight')
print("✓ Saved: revenue_by_genre_bar_chart.png")
plt.close()

# Visualization 2: Market Share Pie Chart
fig, ax = plt.subplots(figsize=(10, 8))

top_10_market = genre_revenue.head(10).copy()
colors = plt.cm.Set3(range(len(top_10_market)))

# Explode the largest slice (Action)
explode = [0.1 if i == 0 else 0 for i in range(len(top_10_market))]

wedges, texts, autotexts = ax.pie(
    top_10_market['market_share_pct'],
    labels=top_10_market.index,
    autopct='%1.1f%%',
    startangle=90,
    colors=colors,
    explode=explode,
    shadow=True
)

# Style the text
for text in texts:
    text.set_fontsize(11)
    text.set_fontweight('bold')

for autotext in autotexts:
    autotext.set_color('white')
    autotext.set_fontsize(10)
    autotext.set_fontweight('bold')

ax.set_title(
    'Market Share by Genre (Top 10)',
    fontsize=14,
    fontweight='bold',
    pad=20
)

plt.tight_layout()
plt.savefig('market_share_by_genre_pie_chart.png', dpi=300, bbox_inches='tight')
print("✓ Saved: market_share_by_genre_pie_chart.png")
plt.close()

# Visualization 3: Revenue Efficiency (Revenue per Game)
fig, ax = plt.subplots(figsize=(12, 8))

efficiency_viz = efficiency_sorted.head(10).copy()
efficiency_viz = efficiency_viz.sort_values('revenue_per_game_million')

ax.barh(
    efficiency_viz.index,
    efficiency_viz['revenue_per_game_million'],
    color='coral'
)

ax.set_xlabel('Revenue per Game (₹ Millions)', fontsize=12, fontweight='bold')
ax.set_ylabel('Genre', fontsize=12, fontweight='bold')
ax.set_title(
    'Revenue Efficiency by Genre (Top 10)',
    fontsize=14,
    fontweight='bold',
    pad=20
)

# Add value labels
for i, v in enumerate(efficiency_viz['revenue_per_game_million']):
    ax.text(v + 10, i, f'₹{v:.0f}M', va='center', fontsize=10)

plt.tight_layout()
plt.savefig('revenue_efficiency_by_genre.png', dpi=300, bbox_inches='tight')
print("✓ Saved: revenue_efficiency_by_genre.png")
plt.close()

print("\n" + "=" * 70)
print("✓ ANALYSIS COMPLETE!")
print("=" * 70)
print("\nFiles created:")
print("  • genre_revenue_analysis.csv")
print("  • top_revenue_games.csv")
print("  • revenue_executive_summary.csv")
print("  • revenue_by_genre_bar_chart.png")
print("  • market_share_by_genre_pie_chart.png")
print("  • revenue_efficiency_by_genre.png")
print("\nThese files are ready for:")
print("  → BigQuery upload")
print("  → Looker Studio dashboards")
print("  → GitHub portfolio")
print("  → Mentor presentation")


CREATING VISUALIZATIONS
✓ Saved: revenue_by_genre_bar_chart.png
✓ Saved: market_share_by_genre_pie_chart.png
✓ Saved: revenue_efficiency_by_genre.png

✓ ANALYSIS COMPLETE!

Files created:
  • genre_revenue_analysis.csv
  • top_revenue_games.csv
  • revenue_executive_summary.csv
  • revenue_by_genre_bar_chart.png
  • market_share_by_genre_pie_chart.png
  • revenue_efficiency_by_genre.png

These files are ready for:
  → BigQuery upload
  → Looker Studio dashboards
  → GitHub portfolio
  → Mentor presentation
